In [0]:
# Databricks notebook source
# tutorial_name: 02-Day7-Delta-Live-Tables
# notebookName: 02-Day7-Delta-Live-Tables
# COMMAND ----------

# DBTITLE 1,Paths
notepoint_rel = "hands-on/day-07/notebooks/02-Day7-Delta-Live-Tables.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY7_PREFIX = f"{BASE_PATH}day07-{STUDENT_ID}"
# COMMAND ----------

# DBTITLE 1,Prerequisite check
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Please complete Day 5 notebook 01 first.")

print("DAY7_PREFIX (for any sink paths you add):", DAY7_PREFIX)
# COMMAND ----------


## Day 7 — Item 20: Delta Live Tables (DLT)

Syllabus item 20 — what DLT is; declarative pipelines; **`LIVE TABLE`** / **`STREAMING LIVE TABLE`**; **expectations** with violation policy; **continuous vs triggered**; short demo.

Focus: bronze → silver → gold using `@dlt.table` / `@dlt.view`. In SQL pipelines you declare `CREATE LIVE TABLE` (batch) or `CREATE STREAMING LIVE TABLE` (`readStream` inside); here bronze reads **batch** Delta from Day 5 (`P_BASIC`) so the class does not depend on a live file stream.

**Expectations — course SQL shape (reference):**

```sql
CONSTRAINT valid_count
EXPECT (count >= 0)
ON VIOLATION DROP ROW
```

The gold table below uses **`@dlt.expect_or_drop`** — same **drop row** policy as `ON VIOLATION DROP ROW`. Other Python decorators: `@dlt.expect` (log / retain bad rows per policy), `@dlt.expect_or_fail` (fail the update).

**Triggered vs continuous:** set in the **pipeline** UI. Use **Triggered** for most class demos (scheduled or manual full runs). **Continuous** keeps a long-running cluster consuming streams (higher cost; use when you teach near–real-time).

**Steps:** Workspace → **Delta Live Tables** / **Workflows** → **Create pipeline** → **Notebook** = this file → **Target schema** (e.g. `day07_<your_id>`) → **Start**.

Extra concept labs (streaming `@dlt.table` with `readStream`) live in optional bundles such as `databricks/.../Working_with_Delta_Live_Tables.md` — extend only if time allows.


In [0]:
import dlt
from pyspark.sql.functions import col


@dlt.table(comment="Bronze LIVE (batch): read Day 5 flight_summary_basic Delta path")
def bronze_flights():
    return spark.read.format("delta").load(P_BASIC)


@dlt.view()
def silver_flights():
    return dlt.read("bronze_flights").filter(col("count").isNotNull())


@dlt.table(comment="Gold: expectation with DROP ROW policy (course README)")
@dlt.expect_or_drop("count_non_negative", "count >= 0")
def gold_flights():
    return dlt.read("silver_flights")


### Notes

- If `import dlt` fails, you are not running inside a DLT pipeline context; create and start a pipeline as above.
- Stack more `@dlt.expect_or_drop` / `@dlt.expect` decorators on `gold_flights` or add another `@dlt.table` that uses `spark.readStream` to demonstrate **STREAMING LIVE** (separate pipeline or extra notebook recommended).
- This folder covers streaming and DLT (Day 7). Jobs, SQL warehouses, and monitoring appear in `hands-on/day-08/` and `hands-on/day-09/`.
